# Downloading the Dataset from Kaggle
> Downloading the cats and dogs dataset

> Kaggle provides the download command on its website.


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("bhavikjikadara/dog-and-cat-classification-dataset")

print("Path to dataset files:", path)

## Exploring the Dataset

In [ ]:
import os

data_dir = os.path.join(path, "PetImages")

dog_path = os.path.join(data_dir, 'Dog')
cat_path = os.path.join(data_dir, 'Cat')

dog_imgs = os.listdir(dog_path)
cat_imgs = os.listdir(cat_path)

print(f"Cat images: {len(dog_imgs)}")
print(f"Dog images: {len(cat_imgs)}")
print(f"Total images: {len(dog_imgs) + len(cat_imgs)}")

print(f"First 5 cat images: {cat_imgs[:5]}")
print(f"First 5 dog images: {dog_imgs[:5]}")

dog_imgs = [os.path.join(dog_path, img) for img in dog_imgs]
cat_imgs = [os.path.join(cat_path, img) for img in cat_imgs]

print(f"First cat image: {cat_imgs[0]}")
print(f"First dog image: {dog_imgs[0]}")

## Splitting the Dataset

In [ ]:
import tensorflow as tf
from tensorflow import keras

train_ds, tmp_ds = keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    image_size=(128,128),
    seed=123,
    subset="both",
    batch_size=32
)

class_names = train_ds.class_names
print(class_names)



In [ ]:
val_ds = tmp_ds.take(100)
test_ds = tmp_ds.skip(100)

train_ds = train_ds.apply(tf.data.experimental.ignore_errors())
tmp_ds = tmp_ds.apply(tf.data.experimental.ignore_errors())

# Smaller subsets for faster experimentation
train_ds_fast = train_ds.take(300)
val_ds_fast = val_ds.take(60)
test_ds_fast = test_ds.take(30)

## Data augmentation

In [5]:
from tensorflow.keras import layers, models

data_augmentation = keras.Sequential([
  layers.RandomFlip("horizontal"),            # Flip horizontally
  layers.RandomRotation(0.1),                 # Rotate slightly (up to about 36 degrees)
  layers.RandomZoom(0.3),                     # Apply a slight zoom in or out
  layers.RandomTranslation(0.3, 0.5),         # Shift the image horizontally and vertically
  layers.RandomContrast(0.3)                  # Adjust the image contrast
])

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 10))

# Get the first image from the first training batch
for images, _ in train_ds.take(1):
  primeira_imagem = images[0]

  # Generate nine augmented versions of the image.
  for i in range(9):
    ax = plt.subplot(3, 3, i + 1)
    # Apply augmentation (training=True enables the transformation).
    imagem_mutante = data_augmentation(tf.expand_dims(primeira_imagem, 0), training=True)

    plt.imshow(imagem_mutante[0].numpy().astype("uint8"))
    plt.axis("off")

plt.show()

## Building the Convolutional Neural Network (CNN)

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(128, 128, 3), # Use the same dimensions as before.
    include_top=False,         # Remove the classification head from the original network.
    weights='imagenet'         # Load pretrained weights.
)

base_model.trainable = False

In [ ]:


model = models.Sequential([
  # Previous approach
  # layers.Flatten(input_shape=(28,28)) # Flattening layer
  # layers.Dense(512, activation=tensorflow.nn.relu), # Neural network layers

  # Current approach
  # layers.Input(shape=(img_height, img_width, 3)), # Set the input dimensions for the network
  # layers.Rescaling(1/255.0), # Rescale pixel values
  # layers.Conv2D(32, (3, 3), activation='relu', padding='same'), # Scan the image for 32 different patterns
  # layers.MaxPooling2D((2, 2)), # Downsample the image while retaining important features
  # layers.BatchNormalization(), # Keep values balanced to help the model learn
  # layers.Dropout(0.25), # Randomly disable neurons to reduce overfitting

  layers.Input(shape=(128, 128, 3)),

  data_augmentation,

  # layers.Rescaling(1./255), # For another model (range: 0 to 1)

  # layers.Conv2D(16, 3, padding='same', activation='relu'),
  # layers.MaxPooling2D(),

  # layers.Conv2D(32, 3, padding='same', activation='relu'),
  # layers.MaxPooling2D(),

  # layers.Conv2D(64, 3, padding='same', activation='relu'),
  # layers.MaxPooling2D(),

  # layers.Flatten(),

  layers.Rescaling(1./127.5, offset=-1), # MobileNetV2 (range: -1 to 1)

  base_model, # Mobile NetV2

  layers.GlobalAveragePooling2D(),

  layers.Dense(128, activation='relu'),
  layers.BatchNormalization(),
  layers.Dropout(0.2),

  layers.Dense(2, activation='softmax')
])

model.summary()

## Compiling and Training the Model

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


train_stats = model.fit(
  train_ds_fast,
  validation_data=val_ds_fast,
  epochs=1,
  batch_size=32
)

In [ ]:
print(f"Final training loss: {train_stats.history['loss'][-1]}")
print(f"Final training accuracy: {train_stats.history['accuracy'][-1]}")

print(f"Final validation loss: {train_stats.history['val_loss'][-1]}")
print(f"Final validation accuracy: {train_stats.history['val_accuracy'][-1]}")

test_loss, test_accuracy = model.evaluate(test_ds_fast)

print(f"Test loss: {test_loss}")
print(f"Test accuracy: {test_accuracy}")

## Testing with Images from the Web

In [ ]:
import numpy as np
img_url = "https://blog-static.petlove.com.br/wp-content/uploads/2019/06/shutterstock_632318627.jpg"

img_path = tf.keras.utils.get_file(origin=img_url)

img = tf.keras.utils.load_img(
    img_path, target_size=(128, 128)
)

plt.imshow(img)
plt.show()

img_array = tf.keras.utils.img_to_array(img)
img_array = tf.expand_dims(img_array, 0)
img_array = img_array / 255.0

predictions = model.predict(img_array)

predicted_class = class_names[np.argmax(predictions)]
confidence = 100 * np.max(predictions)

print(f"\nModel prediction: {predicted_class} (with {confidence:.2f}% confidence!)")